# Country dimension loader

Loads `dim_country` from the World Bank country endpoint and attaches each
country's local currency.

`dim_currency` is not loaded here: it is static ISO 4217 reference data with no
API source, so it is seeded directly in `project_1.sql`.

Why this notebook exists at all: `gdp` and `population` are keyed on country,
`exchange_rates` is keyed on currency, and nothing joined the two. The task asks
for "the keys that link GDP and population statistics to specific currency
exchange rates" - `dim_country.local_currency` is that key. It is also what makes
"GDP in local currency" possible, not just "GDP in a currency the user picked".

The notebook:
- retrieves World Bank country metadata (name, region, income group)
- excludes aggregate entities such as World, regions and income groups
- maps each country to its local currency from `COUNTRY_CURRENCY`
- stores NULL rather than breaking `fk_country_currency` when the FX provider
  does not quote that currency, and reports every such country
- uses an upsert so the notebook can be rerun safely
- prints a coverage report after loading

Shared connection, retry and load helpers live in `etl.py`.

Run order: `project_1.sql` -> **this notebook** -> `gdp_world_bank.ipynb` ->
`population_world_bank.ipynb` -> `api.ipynb`.

MySQL credentials are read from the existing `.env` file.

In [1]:
%pip install -q requests python-dotenv mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import etl


# ============================================================
# COUNTRY -> LOCAL CURRENCY
#
# Grouped by monetary union rather than alphabetically, because the
# groups are the part that carries information: 26 countries share
# the euro, 14 share a CFA franc, and 14 more simply use the US
# dollar. A flat A-Z list would hide all of that.
#
# South Sudan is deliberately absent: its currency SSP is not quoted
# by the FX provider, so the honest value is NULL. The loader reports
# every country it could not map instead of guessing.
# ============================================================

COUNTRY_CURRENCY = {
    # Euro - members plus non-members that use it
    "AND": "EUR", "AUT": "EUR", "BEL": "EUR", "CYP": "EUR", "DEU": "EUR",
    "ESP": "EUR", "EST": "EUR", "FIN": "EUR", "FRA": "EUR", "GRC": "EUR",
    "HRV": "EUR", "IRL": "EUR", "ITA": "EUR", "LTU": "EUR", "LUX": "EUR",
    "LVA": "EUR", "MAF": "EUR", "MCO": "EUR", "MLT": "EUR", "MNE": "EUR",
    "NLD": "EUR", "PRT": "EUR", "SMR": "EUR", "SVK": "EUR", "SVN": "EUR",
    "XKX": "EUR",

    # West African CFA franc (UEMOA)
    "BEN": "XOF", "BFA": "XOF", "CIV": "XOF", "GNB": "XOF", "MLI": "XOF",
    "NER": "XOF", "SEN": "XOF", "TGO": "XOF",

    # Central African CFA franc (CEMAC)
    "CAF": "XAF", "CMR": "XAF", "COG": "XAF", "GAB": "XAF", "GNQ": "XAF",
    "TCD": "XAF",

    # East Caribbean dollar (ECCU)
    "ATG": "XCD", "DMA": "XCD", "GRD": "XCD", "KNA": "XCD", "LCA": "XCD",
    "VCT": "XCD",

    # CFP franc
    "NCL": "XPF", "PYF": "XPF",

    # Caribbean guilder - replaced the Antillean guilder in 2025
    "CUW": "XCG", "SXM": "XCG",

    # US dollar in circulation, no separate national currency
    "ASM": "USD", "ECU": "USD", "FSM": "USD", "GUM": "USD", "MHL": "USD",
    "MNP": "USD", "PLW": "USD", "PRI": "USD", "SLV": "USD", "TCA": "USD",
    "TLS": "USD", "USA": "USD", "VGB": "USD", "VIR": "USD",

    # Australian dollar
    "AUS": "AUD", "KIR": "AUD", "NRU": "AUD", "TUV": "AUD",

    # Danish krone
    "DNK": "DKK", "FRO": "DKK", "GRL": "DKK",

    # Swiss franc
    "CHE": "CHF", "LIE": "CHF",

    # Sterling area - the Isle of Man issues its own notes (IMP),
    # the Channel Islands are reported here against GBP
    "GBR": "GBP", "CHI": "GBP", "IMN": "IMP", "GIB": "GIP",

    # Israeli shekel - also the currency of the West Bank and Gaza
    "ISR": "ILS", "PSE": "ILS",

    # One country, one currency
    "ABW": "AWG", "AFG": "AFN", "AGO": "AOA", "ALB": "ALL", "ARE": "AED",
    "ARG": "ARS", "ARM": "AMD", "AZE": "AZN", "BDI": "BIF", "BGD": "BDT",
    "BGR": "BGN", "BHR": "BHD", "BHS": "BSD", "BIH": "BAM", "BLR": "BYN",
    "BLZ": "BZD", "BMU": "BMD", "BOL": "BOB", "BRA": "BRL", "BRB": "BBD",
    "BRN": "BND", "BTN": "BTN", "BWA": "BWP", "CAN": "CAD", "CHL": "CLP",
    "CHN": "CNY", "COD": "CDF", "COL": "COP", "COM": "KMF", "CPV": "CVE",
    "CRI": "CRC", "CUB": "CUP", "CYM": "KYD", "CZE": "CZK", "DJI": "DJF",
    "DOM": "DOP", "DZA": "DZD", "EGY": "EGP", "ERI": "ERN", "ETH": "ETB",
    "FJI": "FJD", "GEO": "GEL", "GHA": "GHS", "GIN": "GNF", "GMB": "GMD",
    "GTM": "GTQ", "GUY": "GYD", "HKG": "HKD", "HND": "HNL", "HTI": "HTG",
    "HUN": "HUF", "IDN": "IDR", "IND": "INR", "IRN": "IRR", "IRQ": "IQD",
    "ISL": "ISK", "JAM": "JMD", "JOR": "JOD", "JPN": "JPY", "KAZ": "KZT",
    "KEN": "KES", "KGZ": "KGS", "KHM": "KHR", "KOR": "KRW", "KWT": "KWD",
    "LAO": "LAK", "LBN": "LBP", "LBR": "LRD", "LBY": "LYD", "LKA": "LKR",
    "LSO": "LSL", "MAC": "MOP", "MAR": "MAD", "MDA": "MDL", "MDG": "MGA",
    "MDV": "MVR", "MEX": "MXN", "MKD": "MKD", "MMR": "MMK", "MNG": "MNT",
    "MOZ": "MZN", "MRT": "MRU", "MUS": "MUR", "MWI": "MWK", "MYS": "MYR",
    "NAM": "NAD", "NGA": "NGN", "NIC": "NIO", "NOR": "NOK", "NPL": "NPR",
    "NZL": "NZD", "OMN": "OMR", "PAK": "PKR", "PAN": "PAB", "PER": "PEN",
    "PHL": "PHP", "PNG": "PGK", "POL": "PLN", "PRK": "KPW", "PRY": "PYG",
    "QAT": "QAR", "ROU": "RON", "RUS": "RUB", "RWA": "RWF", "SAU": "SAR",
    "SDN": "SDG", "SGP": "SGD", "SLB": "SBD", "SLE": "SLE", "SOM": "SOS",
    "SRB": "RSD", "STP": "STN", "SUR": "SRD", "SWE": "SEK", "SWZ": "SZL",
    "SYC": "SCR", "SYR": "SYP", "THA": "THB", "TJK": "TJS", "TKM": "TMT",
    "TON": "TOP", "TTO": "TTD", "TUN": "TND", "TUR": "TRY", "TZA": "TZS",
    "UGA": "UGX", "UKR": "UAH", "URY": "UYU", "UZB": "UZS", "VEN": "VES",
    "VNM": "VND", "VUT": "VUV", "WSM": "WST", "YEM": "YER", "ZAF": "ZAR",
    "ZMB": "ZMW", "ZWE": "ZWL",
}


# ============================================================
# UPSERT
#
# country_iso3 is the primary key, so a rerun refreshes names,
# regions and currencies instead of duplicating rows.
# ============================================================

INSERT_QUERY = """
INSERT INTO dim_country
    (country_iso3, country_name, region, income_group, local_currency)
VALUES
    (%s, %s, %s, %s, %s) AS new
ON DUPLICATE KEY UPDATE
    country_name   = new.country_name,
    region         = new.region,
    income_group   = new.income_group,
    local_currency = new.local_currency
"""

print("Countries mapped in COUNTRY_CURRENCY:", len(COUNTRY_CURRENCY))

Countries mapped in COUNTRY_CURRENCY: 216


In [3]:
def build_rows(countries, known_currencies):
    """
    Shape API rows into (iso3, name, region, income_group, local_currency)
    tuples, and collect the countries left without a currency.

    A code that dim_currency does not hold becomes NULL rather than a foreign
    key violation - the FX provider simply never quoted it.
    """
    rows = []
    unmapped = []

    for country in countries:
        iso3 = country["id"]
        currency = COUNTRY_CURRENCY.get(iso3)

        if currency not in known_currencies:
            if currency is None:
                unmapped.append(f"{iso3} - no mapping")
            else:
                unmapped.append(f"{iso3} - mapped to {currency}, absent from dim_currency")
            currency = None

        rows.append(
            (
                iso3,
                country["name"],
                (country.get("region") or {}).get("value"),
                (country.get("incomeLevel") or {}).get("value"),
                currency,
            )
        )

    return rows, unmapped

In [4]:
# ============================================================
# EXTRACT -> TRANSFORM -> LOAD
# ============================================================

session = etl.make_session()
connection = etl.connect_mysql()

try:
    known_currencies = etl.load_currency_keys(connection)
    print("Currency codes available in dim_currency:", len(known_currencies))

    if not known_currencies:
        raise RuntimeError(
            "dim_currency is empty - run project_1.sql before this notebook."
        )

    # ----------------------------
    # EXTRACT
    # ----------------------------
    countries = etl.fetch_worldbank_countries(session)
    print("Real countries/economies returned by API:", len(countries))

    # ----------------------------
    # TRANSFORM
    # ----------------------------
    rows, unmapped = build_rows(countries, known_currencies)

    # ----------------------------
    # LOAD
    # ----------------------------
    loaded = etl.upsert(connection, INSERT_QUERY, rows)
    print("\ndim_country load completed successfully. Rows processed:", loaded)

    if unmapped:
        print("\nStored with local_currency = NULL:")
        for item in unmapped:
            print("  ", item)
    else:
        print("\nEvery country mapped to a currency.")

except Exception:
    print("\ndim_country load failed.")
    raise

finally:
    connection.close()
    print("MySQL connection closed.")

Currency codes available in dim_currency: 172
Real countries/economies returned by API: 217

dim_country load completed successfully. Rows processed: 217

Stored with local_currency = NULL:
   SSD - no mapping
MySQL connection closed.


In [5]:
# ============================================================
# VALIDATION
# ============================================================

connection = etl.connect_mysql()

try:
    print("dim_country summary:")
    etl.print_rows(connection, """
        SELECT
            COUNT(*)                        AS countries,
            COUNT(local_currency)           AS with_currency,
            COUNT(*) - COUNT(local_currency) AS without_currency,
            COUNT(DISTINCT local_currency)  AS distinct_currencies,
            COUNT(DISTINCT region)          AS regions
        FROM dim_country
    """)

    # Currency unions are the interesting rows: one currency, many countries.
    print()
    print("Shared currencies (monetary unions and dollarised economies):")
    etl.print_rows(connection, """
        SELECT local_currency, COUNT(*) AS countries
        FROM dim_country
        WHERE local_currency IS NOT NULL
        GROUP BY local_currency
        HAVING COUNT(*) > 1
        ORDER BY countries DESC, local_currency
    """)

    # Every country must resolve to a currency the FX table actually quotes,
    # otherwise "GDP in local currency" would silently return nothing.
    print()
    print("Countries whose local currency has no rate in exchange_rates:")
    etl.print_rows(connection, """
        SELECT c.country_iso3, c.country_name, c.local_currency
        FROM dim_country AS c
        LEFT JOIN exchange_rates AS r
               ON r.target_currency = c.local_currency
        WHERE c.local_currency IS NOT NULL
          AND r.target_currency IS NULL
        GROUP BY c.country_iso3, c.country_name, c.local_currency
        ORDER BY c.country_iso3
    """)

finally:
    connection.close()

dim_country summary:
   (217, 216, 1, 151, 7)

Shared currencies (monetary unions and dollarised economies):
   ('EUR', 26)
   ('USD', 14)
   ('XOF', 8)
   ('XAF', 6)
   ('XCD', 6)
   ('AUD', 4)
   ('DKK', 3)
   ('CHF', 2)
   ('GBP', 2)
   ('ILS', 2)
   ('XCG', 2)
   ('XPF', 2)

Countries whose local currency has no rate in exchange_rates:
